In [1]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week8-assignment-1"). \
config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [2]:
from pyspark.sql import *

### Demonstrate the following :- 
### Running total, Grouping aggregates and various Window functions like rank, dense_rank, row_num, lead, lag.- Also try creating a pivot view.

# 1 Running total 

In [3]:
windowDF = spark.read.format("csv").option("header","true").option("inferSchema","true").load("/public/trendytech/datasets/windowdatamodified.csv")

In [4]:
windowDF.show(3)

+---------+-------+-----------+-------------+------------+
|  country|weeknum|numinvoices|totalquantity|invoicevalue|
+---------+-------+-----------+-------------+------------+
|    Spain|     49|          1|           67|      174.72|
|  Germany|     48|         11|         1795|      1600.0|
|Lithuania|     48|          3|          622|     1598.06|
+---------+-------+-----------+-------------+------------+
only showing top 3 rows



In [5]:
window = Window.partitionBy("country").orderBy("weeknum").rowsBetween(Window.unboundedPreceding,Window.currentRow)

In [6]:
newDF = windowDF.withColumn("running_tot", sum("totalquantity").over(window))

In [7]:
newDF.show(10)

+-------+-------+-----------+-------------+------------+-----------+
|country|weeknum|numinvoices|totalquantity|invoicevalue|running_tot|
+-------+-------+-----------+-------------+------------+-----------+
| Sweden|     50|          3|         3714|      2646.3|       3714|
|Germany|     48|         11|         1795|      1600.0|       1795|
|Germany|     49|         12|         1852|      1800.0|       3647|
|Germany|     50|         15|         1973|      1800.0|       5620|
|Germany|     51|          5|         1103|      1600.0|       6723|
| France|     48|          4|         1299|       500.0|       1299|
| France|     49|          9|         2303|       500.0|       3602|
| France|     50|          6|          529|      537.32|       4131|
| France|     51|          5|          847|       500.0|       4978|
|Belgium|     48|          1|          528|       800.0|        528|
+-------+-------+-----------+-------------+------------+-----------+
only showing top 10 rows



In [8]:
windowDF.printSchema()

root
 |-- country: string (nullable = true)
 |-- weeknum: integer (nullable = true)
 |-- numinvoices: integer (nullable = true)
 |-- totalquantity: integer (nullable = true)
 |-- invoicevalue: double (nullable = true)



# 2 group by aggregations

In [9]:
groupagg = windowDF.groupBy("country") \
.agg(sum("numinvoices").alias("tot_inv") , max("invoicevalue").alias("max_inv"),min("invoicevalue").alias("min_inv"))

In [10]:
groupagg.show(5) ## total number of invoices per country, max and min invoice amounts

+-------+-------+-------+-------+
|country|tot_inv|max_inv|min_inv|
+-------+-------+-------+-------+
| Sweden|      3| 2646.3| 2646.3|
|Germany|     43| 1800.0| 1600.0|
| France|     24| 537.32|  500.0|
|Belgium|      5|  800.0| 625.16|
|Finland|      1|  892.8|  892.8|
+-------+-------+-------+-------+
only showing top 5 rows



# 3 rank, dense_rank and row_number ()

In [11]:
windowDF.show(10)

+---------+-------+-----------+-------------+------------+
|  country|weeknum|numinvoices|totalquantity|invoicevalue|
+---------+-------+-----------+-------------+------------+
|    Spain|     49|          1|           67|      174.72|
|  Germany|     48|         11|         1795|      1600.0|
|Lithuania|     48|          3|          622|     1598.06|
|  Germany|     49|         12|         1852|      1800.0|
|  Bahrain|     51|          1|           54|      205.74|
|  Iceland|     49|          1|          319|      711.79|
|    India|     51|          5|           95|       300.0|
|Australia|     50|          2|          133|      387.95|
|    Italy|     49|          1|           -2|       -17.0|
|    India|     49|          5|         1280|      3284.1|
+---------+-------+-----------+-------------+------------+
only showing top 10 rows



In [12]:
window1 = Window.partitionBy("country").orderBy(("invoicevalue"))

In [13]:
newDF1 = windowDF.withColumn("rank", rank().over(window1)).withColumn("dense_rank", dense_rank().over(window1)).withColumn("row_num", row_number().over(window1))

In [14]:
newDF1.show(20)

+-------+-------+-----------+-------------+------------+----+----------+-------+
|country|weeknum|numinvoices|totalquantity|invoicevalue|rank|dense_rank|row_num|
+-------+-------+-----------+-------------+------------+----+----------+-------+
| Sweden|     50|          3|         3714|      2646.3|   1|         1|      1|
|Germany|     48|         11|         1795|      1600.0|   1|         1|      1|
|Germany|     51|          5|         1103|      1600.0|   1|         1|      2|
|Germany|     49|         12|         1852|      1800.0|   3|         2|      3|
|Germany|     50|         15|         1973|      1800.0|   3|         2|      4|
| France|     51|          5|          847|       500.0|   1|         1|      1|
| France|     49|          9|         2303|       500.0|   1|         1|      2|
| France|     48|          4|         1299|       500.0|   1|         1|      3|
| France|     50|          6|          529|      537.32|   4|         2|      4|
|Belgium|     50|          2

# 4 lead and lag

In [15]:
lag_window = Window.partitionBy("country").orderBy("weeknum")

In [16]:
newDF2 = windowDF.withColumn("prev_week", lag("weeknum").over(lag_window)).withColumn("next_week", lead("weeknum").over(lag_window))

In [17]:
newDF2.show()

+-------+-------+-----------+-------------+------------+---------+---------+
|country|weeknum|numinvoices|totalquantity|invoicevalue|prev_week|next_week|
+-------+-------+-----------+-------------+------------+---------+---------+
| Sweden|     50|          3|         3714|      2646.3|     null|     null|
|Germany|     48|         11|         1795|      1600.0|     null|       49|
|Germany|     49|         12|         1852|      1800.0|       48|       50|
|Germany|     50|         15|         1973|      1800.0|       49|       51|
|Germany|     51|          5|         1103|      1600.0|       50|     null|
| France|     48|          4|         1299|       500.0|     null|       49|
| France|     49|          9|         2303|       500.0|       48|       50|
| France|     50|          6|          529|      537.32|       49|       51|
| France|     51|          5|          847|       500.0|       50|     null|
|Belgium|     48|          1|          528|       800.0|     null|       50|

# 5 Pivot

In [22]:
windowDF.show()

+--------------+-------+-----------+-------------+------------+
|       country|weeknum|numinvoices|totalquantity|invoicevalue|
+--------------+-------+-----------+-------------+------------+
|         Spain|     49|          1|           67|      174.72|
|       Germany|     48|         11|         1795|      1600.0|
|     Lithuania|     48|          3|          622|     1598.06|
|       Germany|     49|         12|         1852|      1800.0|
|       Bahrain|     51|          1|           54|      205.74|
|       Iceland|     49|          1|          319|      711.79|
|         India|     51|          5|           95|       300.0|
|     Australia|     50|          2|          133|      387.95|
|         Italy|     49|          1|           -2|       -17.0|
|         India|     49|          5|         1280|      3284.1|
|         Spain|     50|          2|          400|     1049.01|
|United Kingdom|     51|        200|        28782|    75103.46|
|        Norway|     49|          1|    

In [33]:
newDF3 = windowDF.groupBy("country").pivot("weeknum").count() # one row per country, one column per week number, and each cell = the number of rows that country had in that week

In [32]:
newDF3.show()

+---------------+----+----+----+----+
|        country|  48|  49|  50|  51|
+---------------+----+----+----+----+
|         Sweden|null|null|   1|null|
|        Germany|   1|   1|   1|   1|
|         France|   1|   1|   1|   1|
|        Belgium|   1|null|   1|   1|
|        Finland|null|null|   1|null|
|          India|   1|   1|   1|   1|
|          Italy|   1|   1|null|   1|
|      Lithuania|   1|   1|null|null|
|         Norway|   1|   1|null|null|
|          Spain|   1|   1|   1|null|
|        Denmark|null|   1|null|null|
|        Iceland|null|   1|null|null|
|         Israel|null|null|   1|null|
|Channel Islands|null|   1|null|null|
|         Cyprus|null|null|   1|null|
|    Switzerland|   1|null|null|   1|
|          Japan|   1|   1|null|null|
|         Poland|   1|null|null|null|
|       Portugal|   1|   1|   1|null|
|      Australia|   1|   1|   1|null|
+---------------+----+----+----+----+
only showing top 20 rows



In [34]:
newDF_v1 = windowDF.groupBy("country") \
    .pivot("weeknum") \
    .agg(sum("numinvoices")) #each cell shows the actual invoice count for that country/week,

In [26]:
newDF_v1.show()

+---------------+----+----+----+----+
|        country|  48|  49|  50|  51|
+---------------+----+----+----+----+
|         Sweden|null|null|   3|null|
|        Germany|  11|  12|  15|   5|
|         France|   4|   9|   6|   5|
|        Belgium|   1|null|   2|   2|
|        Finland|null|null|   1|null|
|          India|   7|   5|   5|   5|
|          Italy|   1|   1|null|   1|
|      Lithuania|   3|   1|null|null|
|         Norway|   1|   1|null|null|
|          Spain|   1|   1|   2|null|
|        Denmark|null|   1|null|null|
|        Iceland|null|   1|null|null|
|         Israel|null|null|   1|null|
|Channel Islands|null|   1|null|null|
|         Cyprus|null|null|   1|null|
|    Switzerland|   1|null|null|   1|
|          Japan|   1|   2|null|null|
|         Poland|   1|null|null|null|
|       Portugal|   1|   4|   3|null|
|      Australia|   1|   1|   2|null|
+---------------+----+----+----+----+
only showing top 20 rows



In [35]:
newDF_v2 = windowDF.groupBy("country") \
    .pivot("weeknum") \
    .agg(sum("invoicevalue")) #a week-over-week revenue matrix by country,

In [28]:
newDF_v2.show()

+---------------+-------+-------+-------+-------+
|        country|     48|     49|     50|     51|
+---------------+-------+-------+-------+-------+
|         Sweden|   null|   null| 2646.3|   null|
|        Germany| 1600.0| 1800.0| 1800.0| 1600.0|
|         France|  500.0|  500.0| 537.32|  500.0|
|        Belgium|  800.0|   null| 625.16|  800.0|
|        Finland|   null|   null|  892.8|   null|
|          India|  300.0| 3284.1|2321.78|  300.0|
|          Italy|  427.8|  -17.0|   null|  383.7|
|      Lithuania|1598.06|   63.0|   null|   null|
|         Norway|1919.14|1867.98|   null|   null|
|          Spain|  620.0| 174.72|1049.01|   null|
|        Denmark|   null| 1281.5|   null|   null|
|        Iceland|   null| 711.79|   null|   null|
|         Israel|   null|   null|-227.44|   null|
|Channel Islands|   null| 363.53|   null|   null|
|         Cyprus|   null|   null|1590.82|   null|
|    Switzerland|  303.4|   null|   null|1001.52|
|          Japan| 320.08|7384.99|   null|   null|


In [37]:
newDF_v4 = windowDF.groupBy("country") \
    .pivot("weeknum") \
    .agg(
        sum("numinvoices").alias("invcnt"),
        sum("invoicevalue").alias("rev")
    ) #a single wide table combining both volume and revenue per week, per country.  multiple aggregates at once

In [24]:
newDF_v4.show()

+---------------+---------+-------+---------+-------+---------+-------+---------+-------+
|        country|48_invcnt| 48_rev|49_invcnt| 49_rev|50_invcnt| 50_rev|51_invcnt| 51_rev|
+---------------+---------+-------+---------+-------+---------+-------+---------+-------+
|         Sweden|     null|   null|     null|   null|        3| 2646.3|     null|   null|
|        Germany|       11| 1600.0|       12| 1800.0|       15| 1800.0|        5| 1600.0|
|         France|        4|  500.0|        9|  500.0|        6| 537.32|        5|  500.0|
|        Belgium|        1|  800.0|     null|   null|        2| 625.16|        2|  800.0|
|        Finland|     null|   null|     null|   null|        1|  892.8|     null|   null|
|          India|        7|  300.0|        5| 3284.1|        5|2321.78|        5|  300.0|
|          Italy|        1|  427.8|        1|  -17.0|     null|   null|        1|  383.7|
|      Lithuania|        3|1598.06|        1|   63.0|     null|   null|     null|   null|
|         

In [38]:
weeks = [48, 49, 50, 51] #explicit week list

newDF_v5 = windowDF.groupBy("country") \
    .pivot("weeknum", weeks) \
    .agg(sum("invoicevalue"))

In [39]:
newDF_v5.show()

+---------------+-------+-------+-------+-------+
|        country|     48|     49|     50|     51|
+---------------+-------+-------+-------+-------+
|         Sweden|   null|   null| 2646.3|   null|
|        Germany| 1600.0| 1800.0| 1800.0| 1600.0|
|         France|  500.0|  500.0| 537.32|  500.0|
|        Belgium|  800.0|   null| 625.16|  800.0|
|        Finland|   null|   null|  892.8|   null|
|          India|  300.0| 3284.1|2321.78|  300.0|
|          Italy|  427.8|  -17.0|   null|  383.7|
|      Lithuania|1598.06|   63.0|   null|   null|
|         Norway|1919.14|1867.98|   null|   null|
|          Spain|  620.0| 174.72|1049.01|   null|
|        Denmark|   null| 1281.5|   null|   null|
|        Iceland|   null| 711.79|   null|   null|
|         Israel|   null|   null|-227.44|   null|
|Channel Islands|   null| 363.53|   null|   null|
|         Cyprus|   null|   null|1590.82|   null|
|    Switzerland|  303.4|   null|   null|1001.52|
|          Japan| 320.08|7384.99|   null|   null|


In [40]:
newDF_v6 = windowDF.groupBy("weeknum") \
    .pivot("country") \
    .agg(sum("invoicevalue"))

In [41]:
newDF_v6.show()

+-------+---------+-------+-------+-------+---------------+-------+-------+-------+------+-------+-------+-------+-------+-----+-------+---------+-----------+-------+------+--------+-------+------+-----------+--------------+
|weeknum|Australia|Austria|Bahrain|Belgium|Channel Islands| Cyprus|Denmark|Finland|France|Germany|Iceland|  India| Israel|Italy|  Japan|Lithuania|Netherlands| Norway|Poland|Portugal|  Spain|Sweden|Switzerland|United Kingdom|
+-------+---------+-------+-------+-------+---------------+-------+-------+-------+------+-------+-------+-------+-------+-----+-------+---------+-----------+-------+------+--------+-------+------+-----------+--------------+
|     48|   358.25|   null|   null|  800.0|           null|   null|   null|   null| 500.0| 1600.0|   null|  300.0|   null|427.8| 320.08|  1598.06|      192.6|1919.14|248.16|   131.8|  620.0|  null|      303.4|     166116.72|
|     49|    258.9|   null|   null|   null|         363.53|   null| 1281.5|   null| 500.0| 1800.0| 7